# BO Forge mixed-variable LogEI campaign

This notebook demonstrates the v0.4 `CampaignSession` workflow for a mixed-variable single-objective campaign.

The campaign uses continuous, integer, discrete, and categorical variables while keeping public CSV values in user-facing units and labels.

## 1. Setup

The example config uses a mixed-variable campaign. Internally, BO Forge encodes numeric variables into relaxed latent features and categorical variables into one-hot model features, then decodes suggestions back to runnable user-space values.

In [ ]:
from dataclasses import replace
from pathlib import Path
import os
import shutil
import sys

import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

mpl_cache = PROJECT_ROOT / ".matplotlib-cache"
mpl_cache.mkdir(exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(mpl_cache))

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from bo_forge.io import empty_campaign_log
from bo_forge.session import CampaignSession
from bo_forge.suggestions import suggest_next
from bo_forge.transforms import encoded_dimension, encoded_feature_indices, encoded_feature_names

In [ ]:
config_path = PROJECT_ROOT / "configs" / "05_simple_mixed_logei.yaml"
seed_log_path = PROJECT_ROOT / "examples" / "05_simple_mixed_logei_campaign_log.csv"
working_log_path = PROJECT_ROOT / "examples" / "05_simple_mixed_logei_working_log.csv"
latest_suggestion_path = PROJECT_ROOT / "examples" / "05_simple_mixed_logei_latest_suggestions.csv"
report_path = PROJECT_ROOT / "reports" / "05_simple_mixed_campaign_report.txt"
progress_path = PROJECT_ROOT / "reports" / "05_simple_mixed_progress.pdf"
diagnostics_path = PROJECT_ROOT / "reports" / "05_simple_mixed_diagnostics.pdf"
TARGET_OBSERVED_ROWS = 15

shutil.copyfile(seed_log_path, working_log_path)

campaign = CampaignSession.from_files(config_path=config_path, log_path=working_log_path)

## 2. Inspect campaign state

In [ ]:
campaign.validate()
display(campaign.summary())
display(campaign.next_action())
campaign.df

## 3. Inspect internal model encoding

The public campaign log keeps categorical labels such as `MeCN` and `EtOH`. The GP model sees one-hot categorical features instead, so category labels are not treated as ordered scalar values.

In [ ]:
encoding_metadata = pd.DataFrame(
    {
        "feature_index": range(encoded_dimension(campaign.config)),
        "feature_name": encoded_feature_names(campaign.config),
    }
)

display(encoding_metadata)
encoded_feature_indices(campaign.config)

## 4. Compare mixed initial-design methods

The seed log is still below `initial_design_size`, so the first suggestions are mixed-variable Sobol initial-design rows. The second table shows the same initial-design interface using seeded random sampling.

In [ ]:
initial_suggestions = campaign.suggest_next(batch_size=2)
initial_suggestions.to_csv(latest_suggestion_path, index=False)
display(initial_suggestions)

In [ ]:
random_config = replace(
    campaign.config,
    bo=replace(campaign.config.bo, initial_design_method="random"),
)
random_initial_suggestions = suggest_next(
    random_config,
    empty_campaign_log(random_config),
    batch_size=2,
)
display(random_initial_suggestions)

## 5. Synthetic objective

The synthetic yield has smooth curvature in the continuous variables, a discrete/base effect, and a categorical solvent effect.

In [ ]:
def simulate_yield(row):
    loading = float(row["catalyst_loading"])
    time_scaled = (int(row["reaction_time"]) - 10) / 50
    base = float(row["base_equivalents"])
    solvent = row["solvent"]

    solvent_bonus = {"MeCN": 5.0, "EtOH": 2.0, "Water": -3.0}[solvent]
    base_bonus = {0.1: -4.0, 0.2: 2.0, 0.5: 5.0, 1.0: 1.0}[base]
    peak = np.exp(
        -0.5
        * (
            ((loading - 0.12) / 0.035) ** 2
            + ((time_scaled - 0.55) / 0.22) ** 2
        )
    )
    interaction = 4.0 * np.sin(10.0 * loading + 2.0 * time_scaled)
    return float(45.0 + 25.0 * peak + interaction + base_bonus + solvent_bonus)

## 6. Record the initial mixed suggestions

Each row below represents the normal campaign rhythm: append suggestions, run the experiments, mark those same rows observed, and reload the log.

In [ ]:
campaign.append_suggestions(initial_suggestions)

for _, suggestion in initial_suggestions.iterrows():
    campaign.mark_observed(
        row_id=str(suggestion["row_id"]),
        objective_value=simulate_yield(suggestion),
    )

display(campaign.summary())
campaign.df.tail(4)

## 7. Request a mixed qLogEI batch

After the initial design is complete, `suggest_next(batch_size=2)` uses qLogEI in latent mixed-variable space, then decodes and repairs candidates back to runnable user-space rows. In v0.4.1, each qLogEI batch is optimized under one fixed categorical assignment; mixed-category qLogEI batches are deferred.

In [ ]:
bo_suggestions = campaign.suggest_next(batch_size=2)
bo_suggestions.to_csv(latest_suggestion_path, index=False)
display(bo_suggestions)

In [ ]:
campaign.append_suggestions(bo_suggestions)

for _, suggestion in bo_suggestions.iterrows():
    campaign.mark_observed(
        row_id=str(suggestion["row_id"]),
        objective_value=simulate_yield(suggestion),
    )

display(campaign.summary())
campaign.df.tail(4)

## 8. Reports and diagnostics

For mixed-variable campaigns, `plot_diagnostics()` shows progress and variable coverage. It does not show full interaction structure across all variable types.

In [ ]:
target_records = []
while len(campaign.observed_data()) < TARGET_OBSERVED_ROWS:
    remaining = TARGET_OBSERVED_ROWS - len(campaign.observed_data())
    batch_size = min(campaign.config.bo.batch_size, remaining)
    suggestions = campaign.suggest_next(batch_size=batch_size)
    suggestions.to_csv(latest_suggestion_path, index=False)
    campaign.append_suggestions(suggestions)

    for _, suggestion in suggestions.iterrows():
        value = simulate_yield(suggestion)
        campaign.mark_observed(str(suggestion["row_id"]), value)
        target_records.append(
            {
                "source": suggestion["source"],
                "catalyst_loading": float(suggestion["catalyst_loading"]),
                "reaction_time": int(suggestion["reaction_time"]),
                "base_equivalents": float(suggestion["base_equivalents"]),
                "solvent": suggestion["solvent"],
                "yield_score": value,
            }
        )
    campaign.reload()

assert len(campaign.observed_data()) == TARGET_OBSERVED_ROWS

if target_records:
    display(pd.DataFrame(target_records))

display(campaign.summary())
display(campaign.next_action())

In [ ]:
report = campaign.report()
campaign.export_report(report_path)
display(report["summary"])
display(report["best_observation"])

In [ ]:
campaign.plot_progress(save_path=progress_path);

In [ ]:
campaign.plot_diagnostics(save_path=diagnostics_path);